In [10]:
import pandas as pd
import numpy as np

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

massey_raw = pd.read_csv(
    RAW / "MMasseyOrdinals.csv"
)

men_original = pd.read_csv(
    PROCESSED / "men_processed.csv"
)

men_massey = pd.read_csv(
    PROCESSED / "men_massey.csv"
)

men_seeds = pd.read_csv(
    RAW / "MNCAATourneySeeds.csv"
)

print("Massey raw:", massey_raw.shape)
print("Original men's dataset:", men_original.shape)
print("Existing Massey dataset:", men_massey.shape)

print("Total ranking systems:",
      massey_raw["SystemName"].nunique())

display(massey_raw.head())

Massey raw: (5865001, 5)
Original men's dataset: (1449, 8)
Existing Massey dataset: (1449, 11)
Total ranking systems: 197


,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
0,2003,35,SEL,1102,159
1,2003,35,SEL,1103,229
2,2003,35,SEL,1104,12
3,2003,35,SEL,1105,314
4,2003,35,SEL,1106,260


In [11]:
# Only rankings available before the tournament
massey_pre = massey_raw[
    massey_raw["RankingDayNum"] <= 133
].copy()

# If a system ranks a team multiple times during a season,
# retain only its latest eligible ranking.
massey_latest = (
    massey_pre
    .sort_values(
        ["Season", "SystemName",
         "TeamID", "RankingDayNum"]
    )
    .drop_duplicates(
        ["Season", "SystemName", "TeamID"],
        keep="last"
    )
    .reset_index(drop=True)
)

assert not massey_latest.duplicated(
    ["Season", "SystemName", "TeamID"]
).any()

assert (
    massey_latest["OrdinalRank"] > 0
).all()

print("Latest ranking records:", len(massey_latest))
print("Ranking systems:", massey_latest["SystemName"].nunique())
print("Latest day:", massey_latest["RankingDayNum"].max())

Latest ranking records: 458694
Ranking systems: 197
Latest day: 133


In [12]:
# ============================================================
# CELL 3: SELECT MASSEY SYSTEMS USING HISTORICAL COVERAGE
# ============================================================

# 1. Get all unique historical tournament team-season pairs
team1 = men_original[
    ["Season", "Team1"]
].rename(columns={"Team1": "TeamID"})

team2 = men_original[
    ["Season", "Team2"]
].rename(columns={"Team2": "TeamID"})

historical_teams = (
    pd.concat([team1, team2], ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

total_required = len(historical_teams)

print("Historical tournament team-seasons:", total_required)

# ============================================================
# 2. FIND RANKINGS FOR HISTORICAL TOURNAMENT TEAMS
# ============================================================

# massey_latest was created in Cell 2
# It contains the latest ranking at or before Day 133
# for each Season, SystemName, TeamID.

covered = massey_latest.merge(
    historical_teams,
    on=["Season", "TeamID"],
    how="inner",
    validate="many_to_one"
)

# Each SystemName / Season / TeamID should appear once
assert not covered.duplicated(
    ["SystemName", "Season", "TeamID"]
).any()

# ============================================================
# 3. CALCULATE COVERAGE FOR EACH RANKING SYSTEM
# ============================================================

coverage = (
    covered
    .groupby("SystemName")
    .size()
    .reset_index(name="CoveredTeams")
)

coverage["TotalRequired"] = total_required

coverage["CoveragePct"] = (
    coverage["CoveredTeams"]
    / coverage["TotalRequired"]
    * 100
)

coverage = (
    coverage
    .sort_values(
        ["CoveragePct", "SystemName"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# ============================================================
# 4. INSPECT DIFFERENT COVERAGE THRESHOLDS
# ============================================================

print("\nNUMBER OF SYSTEMS AT DIFFERENT COVERAGE THRESHOLDS")
print("-" * 55)

for threshold in [100, 99, 95, 90, 80]:
    count = (
        coverage["CoveragePct"] >= threshold - 1e-10
    ).sum()

    print(f"Coverage >= {threshold}%: {count} systems")

print("\nTOP 50 SYSTEMS BY HISTORICAL COVERAGE")

display(
    coverage.head(50)
)

# ============================================================
# 5. SELECT SYSTEMS WITH AT LEAST 95% COVERAGE
# ============================================================

COVERAGE_THRESHOLD = 95

eligible_systems = (
    coverage.loc[
        coverage["CoveragePct"] >= COVERAGE_THRESHOLD,
        "SystemName"
    ]
    .tolist()
)

assert len(eligible_systems) > 0, (
    "No ranking systems meet the coverage threshold."
)

print("\n" + "=" * 55)
print("SELECTED RANKING SYSTEMS")
print("=" * 55)

print("Coverage threshold:", COVERAGE_THRESHOLD, "%")
print("Number of eligible systems:", len(eligible_systems))

print("\nEligible systems:")
print(sorted(eligible_systems))

# ============================================================
# 6. COMPARE AGAINST ORIGINAL FIVE SYSTEMS
# ============================================================

original_systems = {
    "COL", "DOL", "MOR", "POM", "WLK"
}

selected_systems = set(eligible_systems)

additional_systems = sorted(
    selected_systems - original_systems
)

excluded_original = sorted(
    original_systems - selected_systems
)

print("\nOriginal five systems:")
print(sorted(original_systems))

print("\nAdditional systems selected:")
print(additional_systems)

print("\nOriginal systems excluded:")
print(excluded_original)

print("\nAdditional system count:", len(additional_systems))

Historical tournament team-seasons: 1471

NUMBER OF SYSTEMS AT DIFFERENT COVERAGE THRESHOLDS
-------------------------------------------------------
Coverage >= 100%: 5 systems
Coverage >= 99%: 5 systems
Coverage >= 95%: 8 systems
Coverage >= 90%: 11 systems
Coverage >= 80%: 16 systems

TOP 50 SYSTEMS BY HISTORICAL COVERAGE


,SystemName,CoveredTeams,TotalRequired,CoveragePct
0,COL,1471,1471,100.000000
1,DOL,1471,1471,100.000000
2,MOR,1471,1471,100.000000
3,POM,1471,1471,100.000000
4,WLK,1471,1471,100.000000
5,BIH,1406,1471,95.581237
6,RTH,1404,1471,95.445275
7,WOL,1404,1471,95.445275
8,DUN,1341,1471,91.162475
9,SEL,1338,1471,90.958532



SELECTED RANKING SYSTEMS
Coverage threshold: 95 %
Number of eligible systems: 8

Eligible systems:
['BIH', 'COL', 'DOL', 'MOR', 'POM', 'RTH', 'WLK', 'WOL']

Original five systems:
['COL', 'DOL', 'MOR', 'POM', 'WLK']

Additional systems selected:
['BIH', 'RTH', 'WOL']

Original systems excluded:
[]

Additional system count: 3


In [13]:
# ============================================================
# CELL 4: CALCULATE BROAD MASSEY CONSENSUS RANKINGS
# ============================================================

# 1. Keep only eligible ranking systems
selected_ranks = covered[
    covered["SystemName"].isin(eligible_systems)
].copy()

print("Selected ranking records:", len(selected_ranks))
print("Eligible ranking systems:", len(eligible_systems))

# ============================================================
# 2. CALCULATE TEAM-LEVEL CONSENSUS RANKINGS
# ============================================================

# Some systems may not rank every historical tournament team.
# Therefore, calculate mean/median using available rankings.

broad_team_ranks = (
    selected_ranks
    .groupby(
        ["Season", "TeamID"],
        as_index=False
    )
    .agg(
        BroadMeanRank=("OrdinalRank", "mean"),
        BroadMedianRank=("OrdinalRank", "median"),
        SystemsUsed=("SystemName", "nunique")
    )
)

# ============================================================
# 3. VALIDATE RANKING COVERAGE
# ============================================================

assert not broad_team_ranks.duplicated(
    ["Season", "TeamID"]
).any(), "Duplicate team-season rankings detected."

assert len(broad_team_ranks) == total_required, (
    "Some historical tournament teams have no "
    "ranking from any selected system."
)

assert (
    broad_team_ranks["SystemsUsed"] >= 1
).all(), "Some teams have zero ranking systems."

assert (
    broad_team_ranks["SystemsUsed"] <= len(eligible_systems)
).all(), "SystemsUsed exceeds eligible system count."

assert (
    broad_team_ranks[
        ["BroadMeanRank", "BroadMedianRank"]
    ] > 0
).all().all(), "Invalid non-positive rankings detected."

# ============================================================
# 4. DISPLAY NUMBER OF SYSTEMS PER TEAM
# ============================================================

print("\n" + "=" * 55)
print("CONSENSUS RANKING COVERAGE")
print("=" * 55)

print("Historical tournament team-seasons:", total_required)

print("Eligible ranking systems:", len(eligible_systems))

print("Team-season rows generated:", len(broad_team_ranks))

print("\nNumber of ranking systems used per team:")

display(
    broad_team_ranks["SystemsUsed"].describe()
)

print("\nTeams with the fewest available systems:")

display(
    broad_team_ranks
    .sort_values("SystemsUsed")
    .head(15)
)

# ============================================================
# 5. COVERAGE SUMMARY
# ============================================================

systems_distribution = (
    broad_team_ranks
    .groupby("SystemsUsed")
    .size()
    .reset_index(name="NumberOfTeamSeasons")
    .sort_values("SystemsUsed")
)

systems_distribution["Percentage"] = (
    systems_distribution["NumberOfTeamSeasons"]
    / total_required
    * 100
)

print("\nDistribution of available ranking systems:")

display(systems_distribution)

# ============================================================
# 6. DISPLAY EXAMPLE RANKINGS
# ============================================================

print("\nSample broader consensus rankings:")

display(
    broad_team_ranks.head(10)
)

Selected ranking records: 11569
Eligible ranking systems: 8

CONSENSUS RANKING COVERAGE
Historical tournament team-seasons: 1471
Eligible ranking systems: 8
Team-season rows generated: 1471

Number of ranking systems used per team:


count    1471.000000
mean        7.864718
std         0.456308
min         6.000000
25%         8.000000
50%         8.000000
75%         8.000000
max         8.000000
Name: SystemsUsed, dtype: float64


Teams with the fewest available systems:


,Season,TeamID,BroadMeanRank,BroadMedianRank,SystemsUsed
1138,2021,1155,34.166667,33.0,6
1164,2021,1277,53.166667,56.5,6
1163,2021,1276,3.500000,4.0,6
1162,2021,1268,41.333333,40.0,6
1161,2021,1261,26.000000,28.0,6
1160,2021,1260,21.833333,23.5,6
1159,2021,1251,104.333333,96.5,6
1158,2021,1242,16.333333,16.0,6
1157,2021,1234,7.500000,5.5,6
1156,2021,1233,143.000000,140.0,6



Distribution of available ranking systems:


,SystemsUsed,NumberOfTeamSeasons,Percentage
0,6,67,4.554725
1,7,65,4.418763
2,8,1339,91.026513



Sample broader consensus rankings:


,Season,TeamID,BroadMeanRank,BroadMedianRank,SystemsUsed
0,2003,1104,35.625,36.5,8
1,2003,1112,2.375,2.0,8
2,2003,1113,36.375,36.5,8
3,2003,1120,43.375,44.0,8
4,2003,1122,91.500,76.0,8
5,2003,1139,38.625,29.0,8
6,2003,1140,30.750,30.5,8
7,2003,1141,43.250,43.5,8
8,2003,1143,31.375,29.0,8
9,2003,1153,44.625,45.0,8


In [14]:
# Start with existing Massey matchup data
men_broad = men_massey.copy()

# Recreate our previous log-Massey feature
men_broad["LogMasseyRankDiff"] = (
    np.log(men_broad["Team1MasseyRank"])
    - np.log(men_broad["Team2MasseyRank"])
)

rank_cols = [
    "Season",
    "TeamID",
    "BroadMeanRank",
    "BroadMedianRank"
]

# Team1 ratings
team1_ranks = broad_team_ranks[
    rank_cols
].rename(columns={
    "TeamID": "Team1",
    "BroadMeanRank": "Team1BroadMean",
    "BroadMedianRank": "Team1BroadMedian"
})

# Team2 ratings
team2_ranks = broad_team_ranks[
    rank_cols
].rename(columns={
    "TeamID": "Team2",
    "BroadMeanRank": "Team2BroadMean",
    "BroadMedianRank": "Team2BroadMedian"
})

# Merge rankings for both teams
men_broad = men_broad.merge(
    team1_ranks,
    on=["Season", "Team1"],
    how="left",
    validate="many_to_one",
    sort=False
)

men_broad = men_broad.merge(
    team2_ranks,
    on=["Season", "Team2"],
    how="left",
    validate="many_to_one",
    sort=False
)

# Validate row preservation
assert len(men_broad) == len(men_massey)

assert men_broad[
    [
        "Team1BroadMean",
        "Team2BroadMean",
        "Team1BroadMedian",
        "Team2BroadMedian"
    ]
].notna().all().all()

# New log ranking differences
men_broad["BroadMeanRankDiff"] = (
    men_broad["Team1BroadMean"]
    - men_broad["Team2BroadMean"]
)

men_broad["BroadMedianRankDiff"] = (
    men_broad["Team1BroadMedian"]
    - men_broad["Team2BroadMedian"]
)

# Verify original data was not altered
pd.testing.assert_frame_equal(
    men_broad[men_massey.columns].reset_index(drop=True),
    men_massey.reset_index(drop=True),
    check_exact=True
)

assert np.isfinite(
    men_broad[
        [
            "LogMasseyRankDiff",
            "BroadMeanRankDiff",
            "BroadMedianRankDiff"
        ]
    ].to_numpy()
).all()

# Save experimental dataset
men_broad.to_csv(
    PROCESSED / "men_broad_massey.csv",
    index=False
)

print("Final experimental dataset:", men_broad.shape)

display(
    men_broad[
        [
            "Season",
            "Team1",
            "Team2",
            "LogMasseyRankDiff",
            "BroadMeanRankDiff",
            "BroadMedianRankDiff"
        ]
    ].head()
)

Final experimental dataset: (1449, 18)


,Season,Team1,Team2,LogMasseyRankDiff,BroadMeanRankDiff,BroadMedianRankDiff
0,2003,1411,1421,-0.004781,1.625,-5.5
1,2003,1112,1436,-4.099460,-159.375,-152.5
2,2003,1113,1272,0.514664,16.250,17.0
3,2003,1141,1166,0.573188,21.875,25.0
4,2003,1143,1301,-0.329687,-15.250,-19.5


In [15]:
# ==========================================================
# FEATURES
# ==========================================================

base_features = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

feature_sets = {
    "M1_5SystemRawMean": base_features + [
        "MasseyRankDiff"
    ],

    "M2_5SystemLogMean": base_features + [
        "LogMasseyRankDiff"
    ],

    "M3_BroadRawMean": base_features + [
        "BroadMeanRankDiff"
    ],

    "M4_BroadRawMedian": base_features + [
        "BroadMedianRankDiff"
    ]
}

# ==========================================================
# EXISTING TUNED MEN'S XGBOOST PARAMETERS
# ==========================================================

men_params = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}

validation_years = [2017, 2018, 2019, 2021]

# ==========================================================
# TEMPORAL CV
# ==========================================================

results = []

for experiment_name, features in feature_sets.items():

    for year in validation_years:

        train = men_broad[
            men_broad["Season"] < year
        ]

        valid = men_broad[
            men_broad["Season"] == year
        ]

        assert train["Season"].max() < year

        model = XGBClassifier(**men_params)

        model.fit(
            train[features],
            train["Target"]
        )

        predictions = model.predict_proba(
            valid[features]
        )[:, 1]

        score = brier_score_loss(
            valid["Target"],
            predictions
        )

        results.append({
            "Experiment": experiment_name,
            "ValidationSeason": year,
            "BrierScore": score
        })

results_df = pd.DataFrame(results)

# ==========================================================
# PER-SEASON RESULTS
# ==========================================================

print("PER-SEASON BRIER SCORES")

display(
    results_df.pivot(
        index="ValidationSeason",
        columns="Experiment",
        values="BrierScore"
    ).round(6)
)

# ==========================================================
# MEAN CV RESULTS
# ==========================================================

summary = (
    results_df
    .groupby("Experiment")["BrierScore"]
    .mean()
    .sort_values()
)

print("\nMEAN CV BRIER")
display(summary.to_frame("MeanCVBrier"))

# ==========================================================
# VERIFY THE EXISTING FIVE-SYSTEM LOG BENCHMARK
# ==========================================================

expected = 0.180918

actual = summary["M2_5SystemLogMean"]

print(
    f"\nExisting log-Massey benchmark: "
    f"actual={actual:.6f}, expected={expected:.6f}"
)

assert abs(actual - expected) < 0.000002, (
    "Existing Massey benchmark did not reproduce."
)

print("Existing benchmark reproduced successfully.")

PER-SEASON BRIER SCORES


Experiment,M1_5SystemRawMean,M2_5SystemLogMean,M3_BroadRawMean,M4_BroadRawMedian
ValidationSeason,,,,
2017,0.169810,0.164622,0.174198,0.170761
2018,0.210815,0.214595,0.207358,0.215221
2019,0.155767,0.139790,0.152362,0.153485
2021,0.223153,0.204665,0.225135,0.228490



MEAN CV BRIER


,MeanCVBrier
Experiment,
M2_5SystemLogMean,0.180918
M3_BroadRawMean,0.189763
M1_5SystemRawMean,0.189886
M4_BroadRawMedian,0.191989



Existing log-Massey benchmark: actual=0.180918, expected=0.180918
Existing benchmark reproduced successfully.


In [16]:
# ============================================================
# DIAGNOSE BROAD MASSEY SYSTEM SELECTION
# ============================================================

original_systems = {
    "COL", "DOL", "MOR", "POM", "WLK"
}

print("Number of eligible systems:", len(eligible_systems))
print("Eligible systems:", sorted(eligible_systems))

print("\nAdditional systems beyond original five:")
print(sorted(set(eligible_systems) - original_systems))

print("\nOriginal systems excluded:")
print(sorted(original_systems - set(eligible_systems)))

# Compare actual feature values, not just Brier scores
comparison = men_broad[
    [
        "MasseyRankDiff",
        "BroadMeanRankDiff",
        "BroadMedianRankDiff"
    ]
].copy()

comparison["MeanDifference"] = (
    comparison["BroadMeanRankDiff"]
    - comparison["MasseyRankDiff"]
)

print("\nAre broad mean differences identical to original?")
print(
    np.allclose(
        comparison["MasseyRankDiff"],
        comparison["BroadMeanRankDiff"]
    )
)

print("\nMaximum absolute feature difference:")
print(comparison["MeanDifference"].abs().max())

display(comparison.head(10))

Number of eligible systems: 8
Eligible systems: ['BIH', 'COL', 'DOL', 'MOR', 'POM', 'RTH', 'WLK', 'WOL']

Additional systems beyond original five:
['BIH', 'RTH', 'WOL']

Original systems excluded:
[]

Are broad mean differences identical to original?
False

Maximum absolute feature difference:
29.85


,MasseyRankDiff,BroadMeanRankDiff,BroadMedianRankDiff,MeanDifference
0,-1.2,1.625,-5.5,2.825
1,-154.2,-159.375,-152.5,-5.175
2,14.0,16.250,17.0,2.250
3,17.8,21.875,25.0,4.075
4,-13.2,-15.250,-19.5,-2.050
5,6.0,4.375,4.0,-1.625
6,77.4,75.625,76.0,-1.775
7,-0.8,0.500,2.5,1.300
8,-45.0,-46.500,-44.0,-1.500
9,-92.2,-90.625,-86.0,1.575
